# Modul 0: G1 Standing — kontroler PD i fizyka robota

W tym notebooku poznasz robota **Unitree G1** w symulatorze MuJoCo.

Zobaczysz:
1. Jak robot **stoi** dzieki prostemu kontrolerowi PD
2. Co sie dzieje **bez kontrolera** — robot pada (grawitacja wygrywa!)
3. Jak robot reaguje na **perturbacje** (pchnieccie sila 200N)

To cwiczenie pokazuje, dlaczego potrzebujemy lepszych metod sterowania — a wlasnie do tego sluzy **Reinforcement Learning**.

---
*Kurs: AI & Reinforcement Learning dla robotyki*

In [ ]:
# Instalacja wymaganych pakietow
!pip install -q mujoco mujoco_menagerie numpy matplotlib

In [ ]:
import mujoco
import mujoco_menagerie
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Zaladuj model G1
model_path = Path(mujoco_menagerie.__path__[0]) / "unitree_g1" / "scene.xml"
model = mujoco.MjModel.from_xml_path(str(model_path))
data = mujoco.MjData(model)

print(f"Model zaladowany: {model_path}")
print(f"Wersja MuJoCo: {mujoco.__version__}")

## Model G1 — podstawowe informacje

Robot Unitree G1 to humanoid o wielu stopniach swobody. Zobaczmy jego parametry.

In [ ]:
# Podstawowe informacje o modelu
total_mass = sum(model.body_mass)
n_hinge = sum(1 for i in range(model.njnt) if model.jnt_type[i] == 3)

print(f"=== Model G1 ===")
print(f"Masa calkowita:    {total_mass:.2f} kg")
print(f"Liczba DOF (hinge): {n_hinge}")
print(f"Liczba aktuatorow: {model.nu}")
print(f"Liczba stawow:     {model.njnt}")
print(f"Timestep:          {model.opt.timestep}s")
print(f"Liczba keyframes:  {model.nkey}")

# Keyframes
if model.nkey > 0:
    print(f"\n--- Keyframes ---")
    for i in range(model.nkey):
        name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_KEY, i)
        print(f"  [{i}] {name}")

# Pierwsze 12 aktuatorow
print(f"\n--- Aktuatory (pierwsze 12) ---")
for i in range(min(model.nu, 12)):
    name = mujoco.mj_id2name(model, mujoco.mjtObj.mjOBJ_ACTUATOR, i) or f"act_{i}"
    print(f"  [{i:2d}] {name}")

## Renderowanie 3D w Colab

W Google Colab nie mamy okna 3D (viewer), wiec uzywamy `mujoco.Renderer` do renderowania klatek jako obrazow.

Renderer tworzy obraz RGB z aktualnego stanu symulacji, ktory wyswietlamy za pomoca `matplotlib`.

In [ ]:
# Renderowanie pozy stojacej
if model.nkey > 0:
    mujoco.mj_resetDataKeyframe(model, data, 0)
mujoco.mj_forward(model, data)

renderer = mujoco.Renderer(model, height=480, width=640)
renderer.update_scene(data, camera="track")
img = renderer.render()

plt.figure(figsize=(10, 7))
plt.imshow(img)
plt.axis('off')
plt.title('G1 — poza stojaca (keyframe "stand")')
plt.show()

print(f"Wysokosc robota: {data.qpos[2]:.3f}m")

## Symulacja z kontrolerem PD

Kontroler PD utrzymuje robota w pozycji stojacej.

Wzor: `torque = Kp * (q_target - q) + Kd * (0 - dq)`

- **Kp** — proporcjonalna korekta ("sprezyna" ciagnaca do celu)
- **Kd** — tlumienie ("amortyzator" hamujacy oscylacje)

Symulujemy 3 sekundy i renderujemy klatki co 0.5s.

In [ ]:
# Symulacja z PD — 3 sekundy
model = mujoco.MjModel.from_xml_path(str(model_path))
data = mujoco.MjData(model)

if model.nkey > 0:
    mujoco.mj_resetDataKeyframe(model, data, 0)
mujoco.mj_forward(model, data)

KP = 40.0
KD = 5.0
target_qpos = data.qpos[7:].copy()
duration = 3.0
dt = model.opt.timestep
steps = int(duration / dt)

heights = []
times = []
frames = []
frame_times = []

renderer = mujoco.Renderer(model, height=360, width=480)

for step in range(steps):
    # PD control
    pos_error = target_qpos - data.qpos[7:]
    vel = data.qvel[6:]
    torques = KP * pos_error + KD * (0.0 - vel)
    data.ctrl[:] = np.clip(torques, -100, 100)

    mujoco.mj_step(model, data)

    heights.append(data.qpos[2])
    times.append(data.time)

    # Renderuj klatke co 0.5s
    if abs(data.time % 0.5) < dt or step == 0:
        renderer.update_scene(data, camera="track")
        frames.append(renderer.render().copy())
        frame_times.append(data.time)

# Wykres wysokosci
plt.figure(figsize=(10, 4))
plt.plot(times, heights, 'b-', linewidth=2)
plt.xlabel('Czas [s]')
plt.ylabel('Wysokosc [m]')
plt.title(f'Wysokosc robota z PD (Kp={KP}, Kd={KD})')
plt.grid(True, alpha=0.3)
plt.axhline(y=heights[0], color='r', linestyle='--', alpha=0.5, label='Poczatkowa')
plt.legend()
plt.tight_layout()
plt.show()

# Klatki z symulacji
n_frames = min(len(frames), 6)
fig, axes = plt.subplots(1, n_frames, figsize=(4 * n_frames, 4))
if n_frames == 1:
    axes = [axes]
for i in range(n_frames):
    axes[i].imshow(frames[i])
    axes[i].set_title(f't = {frame_times[i]:.1f}s')
    axes[i].axis('off')
plt.suptitle(f'Symulacja z PD (Kp={KP}, Kd={KD}) — robot stoi!', fontsize=14)
plt.tight_layout()
plt.show()

print(f"Koncowa wysokosc: {heights[-1]:.3f}m")
print(f"Robot {'stoi' if heights[-1] > 0.5 else 'upadl'}!")

## Robot BEZ kontrolera — dlaczego potrzebujemy RL

Co sie stanie, gdy wylaczymy kontroler PD? Ustawiamy `ctrl = 0` (brak momentow na stawach).

Robot jest jak marionetka bez sznurkow — grawitacja robi swoje.

In [ ]:
# Symulacja BEZ kontrolera — robot pada
model = mujoco.MjModel.from_xml_path(str(model_path))
data = mujoco.MjData(model)

if model.nkey > 0:
    mujoco.mj_resetDataKeyframe(model, data, 0)
mujoco.mj_forward(model, data)

duration = 2.0
dt = model.opt.timestep
steps = int(duration / dt)

heights_no_ctrl = []
times_no_ctrl = []
frames_no_ctrl = []
frame_times_no_ctrl = []

renderer = mujoco.Renderer(model, height=360, width=480)

for step in range(steps):
    # BEZ kontrolera!
    data.ctrl[:] = 0.0

    mujoco.mj_step(model, data)

    heights_no_ctrl.append(data.qpos[2])
    times_no_ctrl.append(data.time)

    # Renderuj klatke co 0.4s
    if abs(data.time % 0.4) < dt or step == 0:
        renderer.update_scene(data, camera="track")
        frames_no_ctrl.append(renderer.render().copy())
        frame_times_no_ctrl.append(data.time)

# Klatki — robot pada
n_frames = min(len(frames_no_ctrl), 5)
fig, axes = plt.subplots(1, n_frames, figsize=(4 * n_frames, 4))
if n_frames == 1:
    axes = [axes]
for i in range(n_frames):
    axes[i].imshow(frames_no_ctrl[i])
    axes[i].set_title(f't = {frame_times_no_ctrl[i]:.1f}s')
    axes[i].axis('off')
plt.suptitle('BEZ kontrolera — robot pada!', fontsize=14, color='red')
plt.tight_layout()
plt.show()

# Porownanie wykresow
plt.figure(figsize=(10, 5))
plt.plot(times, heights, 'b-', linewidth=2, label=f'Z PD (Kp={KP}, Kd={KD})')
plt.plot(times_no_ctrl, heights_no_ctrl, 'r-', linewidth=2, label='Bez kontrolera')
plt.xlabel('Czas [s]')
plt.ylabel('Wysokosc [m]')
plt.title('Porownanie: z kontrolerem PD vs bez kontrolera')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Koncowa wysokosc BEZ kontrolera: {heights_no_ctrl[-1]:.3f}m")
print(f"Robot upadl po ~{next((t for t, h in zip(times_no_ctrl, heights_no_ctrl) if h < 0.4), duration):.1f}s")

## Perturbacja — pchnij robota

Sprawdzmy, jak kontroler PD radzi sobie z zewnetrznym zaburzeniem.

Po 1 sekundzie symulacji przykladamy sile **200N** do torsu robota (pchnieccie do przodu).
Sila dziala przez 0.1s.

In [ ]:
# Perturbacja — pchnij robota sila 200N
model = mujoco.MjModel.from_xml_path(str(model_path))
data = mujoco.MjData(model)

if model.nkey > 0:
    mujoco.mj_resetDataKeyframe(model, data, 0)
mujoco.mj_forward(model, data)

KP = 40.0
KD = 5.0
target_qpos = data.qpos[7:].copy()
duration = 3.0
dt = model.opt.timestep
steps = int(duration / dt)

torso_id = mujoco.mj_name2id(model, mujoco.mjtObj.mjOBJ_BODY, "torso_link")
perturb_applied = False

heights_perturb = []
times_perturb = []
frames_perturb = []
frame_times_perturb = []

renderer = mujoco.Renderer(model, height=360, width=480)

for step in range(steps):
    # PD control
    pos_error = target_qpos - data.qpos[7:]
    vel = data.qvel[6:]
    torques = KP * pos_error + KD * (0.0 - vel)
    data.ctrl[:] = np.clip(torques, -100, 100)

    # Perturbacja po 1s
    if data.time > 1.0 and not perturb_applied:
        if torso_id >= 0:
            data.xfrc_applied[torso_id, 0] = 200.0  # 200N do przodu
            perturb_applied = True
            print(f"  t={data.time:.2f}s >>> PERTURBACJA: 200N do przodu!")

    # Wylacz sile po 0.1s
    if perturb_applied and data.time > 1.1:
        data.xfrc_applied[:] = 0

    mujoco.mj_step(model, data)

    heights_perturb.append(data.qpos[2])
    times_perturb.append(data.time)

    # Renderuj klatke co 0.5s
    if abs(data.time % 0.5) < dt or step == 0:
        renderer.update_scene(data, camera="track")
        frames_perturb.append(renderer.render().copy())
        frame_times_perturb.append(data.time)

# Klatki z perturbacja
n_frames = min(len(frames_perturb), 6)
fig, axes = plt.subplots(1, n_frames, figsize=(4 * n_frames, 4))
if n_frames == 1:
    axes = [axes]
for i in range(n_frames):
    axes[i].imshow(frames_perturb[i])
    axes[i].set_title(f't = {frame_times_perturb[i]:.1f}s')
    axes[i].axis('off')
plt.suptitle('Perturbacja 200N po 1s — czy PD da rade?', fontsize=14)
plt.tight_layout()
plt.show()

# Wykres porownawczy
plt.figure(figsize=(10, 5))
plt.plot(times, heights, 'b-', linewidth=2, label='PD bez perturbacji')
plt.plot(times_perturb, heights_perturb, 'orange', linewidth=2, label='PD + perturbacja 200N')
plt.axvline(x=1.0, color='red', linestyle='--', alpha=0.7, label='Moment pchnieccia')
plt.xlabel('Czas [s]')
plt.ylabel('Wysokosc [m]')
plt.title('Wplyw perturbacji na kontroler PD')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

final_h = heights_perturb[-1]
print(f"\nKoncowa wysokosc po perturbacji: {final_h:.3f}m")
if final_h > 0.5:
    print("Robot przetrwal perturbacje — PD utrzymal rownowage!")
else:
    print("Robot upadl — prosty PD nie wystarczyl.")
    print("Wlasnie dlatego potrzebujemy RL — uczy sie reagowac na zaburzenia!")

## Nastepny krok

W tym cwiczeniu zobaczylimy:
- Robot **stoi** dzieki kontrolerowi PD z stalymi wzmocnieniami Kp i Kd
- **Bez kontrolera** robot natychmiast pada — grawitacja jest bezlitosna
- **Perturbacja** moze powalic robota — prosty PD ma ograniczenia

W nastepnym module nauczysz sie **tunowac parametry PD** (Kp, Kd) — to klucz do lepszego sterowania.

Przejdz do: **Modul 1: Tuning kontrolera PD** (`ex1_pd_tuning.ipynb`)